# Préparation des données démographiques de l’INSEE

Ce notebook prépare les données démographiques utilisées dans le projet
d’analyse sociodémographique réalisé avec **Python, pandas, Snowflake et dbt**.

Il transforme le fichier Excel source de l’INSEE en une table au format long,
harmonisée avec les dimensions utilisées dans les données OpenClassrooms :
année, région, genre et tranche d’âge.

> Le fichier Excel source n’est pas publié dans le dépôt.

## 1. Import des bibliothèques

In [1]:
import pandas as pd
from pathlib import Path

## 2. Configuration des fichiers

Le notebook peut être lancé depuis la racine du projet ou depuis le dossier
`notebooks`. Le fichier source doit être placé dans `data/raw`.
Le fichier transformé est exporté dans `data/processed`.

In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

input_path = PROJECT_ROOT / "data" / "raw" / "estim-pop-nreg-sexe-aq-1975-2026.xlsx"
output_dir = PROJECT_ROOT / "data" / "processed"
output_path = output_dir / "insee_population_region_sexe_age_2022_2025_long.csv"

output_dir.mkdir(parents=True, exist_ok=True)

if not input_path.exists():
    raise FileNotFoundError(
        "Fichier INSEE introuvable. Placez-le dans data/raw avant d’exécuter le notebook."
    )

## 3. Paramètres de transformation

In [3]:
# Années conservées, car les données OpenClassrooms couvrent 2022 à 2025
years = [2022, 2023, 2024, 2025]

# Dans le fichier INSEE :
# colonnes 22 à 42 = Hommes
# colonnes 43 à 63 = Femmes
gender_blocks = {
    "Homme": 22,
    "Femme": 43,
}

# Correspondance entre les tranches INSEE et les tranches OpenClassrooms
age_mapping = {
    "20 à 24 ans": "20-24 ans",
    "25 à 29 ans": "25-29 ans",
    "30 à 34 ans": "30-34 ans",
    "35 à 39 ans": "35-39 ans",
    "40 à 44 ans": "40-44 ans",
    "45 à 49 ans": "45-49 ans",
    "50 à 54 ans": "50-54 ans",
    "55 à 59 ans": "55-59 ans",
}

# Tranches INSEE additionnées pour créer « 60 ans ou plus »
age_60_plus = [
    "60 à 64 ans",
    "65 à 69 ans",
    "70 à 74 ans",
    "75 à 79 ans",
    "80 à 84 ans",
    "85 à 89 ans",
    "90 à 94 ans",
    "95 ans et plus",
]

# Régions françaises conservées
allowed_regions = {
    "Auvergne-Rhône-Alpes",
    "Bourgogne-Franche-Comté",
    "Bretagne",
    "Centre-Val-de-Loire",
    "Centre-Val de Loire",
    "Corse",
    "Grand Est",
    "Hauts-de-France",
    "Île-de-France",
    "Normandie",
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Pays de la Loire",
    "Provence-Alpes-Côte d'Azur",
    "DOM",
}

# Harmonisation des libellés de région
region_mapping = {
    "DOM": "DROM",
    "Centre-Val-de-Loire": "Centre-Val de Loire",
}

## 4. Transformation au format long

In [4]:
rows = []

for year in years:
    print(f"Traitement de l'année {year}...")

    # Lecture de la feuille correspondant à l’année
    df = pd.read_excel(input_path, sheet_name=str(year), header=None)

    # Les données régionales commencent après les lignes d’en-tête
    data = df.iloc[5:].copy()
    data = data[data[0].isin(allowed_regions)]

    for _, row in data.iterrows():
        region = region_mapping.get(row[0], row[0])

        for gender, start_col in gender_blocks.items():
            age_labels = list(df.iloc[4, start_col:start_col + 21])
            age_to_col = {
                age_label: start_col + index
                for index, age_label in enumerate(age_labels)
            }

            required_ages = set(age_mapping) | set(age_60_plus)
            missing_ages = required_ages.difference(age_to_col)

            if missing_ages:
                raise ValueError(
                    f"Tranches d'âge absentes pour {year} : {sorted(missing_ages)}"
                )

            # Tranches d’âge détaillées
            for insee_age, oc_age in age_mapping.items():
                population = pd.to_numeric(
                    row[age_to_col[insee_age]],
                    errors="raise",
                )

                rows.append({
                    "YEAR_INSEE": year,
                    "REGION": region,
                    "GENDER": gender,
                    "AGE_GROUP": oc_age,
                    "POPULATION_INSEE": int(population),
                })

            # Agrégation de la tranche « 60 ans ou plus »
            population_60_plus = sum(
                pd.to_numeric(row[age_to_col[age]], errors="raise")
                for age in age_60_plus
            )

            rows.append({
                "YEAR_INSEE": year,
                "REGION": region,
                "GENDER": gender,
                "AGE_GROUP": "60 ans ou plus",
                "POPULATION_INSEE": int(population_60_plus),
            })

Traitement de l'année 2022...
Traitement de l'année 2023...
Traitement de l'année 2024...
Traitement de l'année 2025...


## 5. Création et contrôle du jeu de données final

In [5]:
insee_long = pd.DataFrame(rows)

expected_rows = 4 * 14 * 2 * 9

print("===== Vérifications =====")
print("Nombre de lignes :", len(insee_long))
print("Années :", sorted(insee_long["YEAR_INSEE"].unique()))
print("Nombre de régions :", insee_long["REGION"].nunique())
print("Genres :", sorted(insee_long["GENDER"].unique()))
print("Tranches d'âge :", sorted(insee_long["AGE_GROUP"].unique()))

print("\nNombre de lignes par année :")
print(insee_long.groupby("YEAR_INSEE").size())

print("\nAperçu du fichier final :")
display(insee_long.head())

assert len(insee_long) == expected_rows, (
    f"Nombre de lignes inattendu : {len(insee_long)} au lieu de {expected_rows}."
)

assert not insee_long.isna().any().any(), (
    "Le jeu de données final contient des valeurs manquantes."
)

print("\nContrôles validés.")


===== Vérifications =====
Nombre de lignes : 1008
Années : [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Nombre de régions : 14
Régions : ['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne', 'Centre-Val de Loire', 'Corse', 'DROM', 'Grand Est', 'Hauts-de-France', 'Normandie', 'Nouvelle-Aquitaine', 'Occitanie', 'Pays de la Loire', "Provence-Alpes-Côte d'Azur", 'Île-de-France']
Genres : ['Femme', 'Homme']
Tranches d'âge : ['20-24 ans', '25-29 ans', '30-34 ans', '35-39 ans', '40-44 ans', '45-49 ans', '50-54 ans', '55-59 ans', '60 ans ou plus']

Nombre de lignes par année :
YEAR_INSEE
2022    252
2023    252
2024    252
2025    252
dtype: int64

Aperçu du fichier final :
   YEAR_INSEE                REGION GENDER  AGE_GROUP  POPULATION_INSEE
0        2022  Auvergne-Rhône-Alpes  Homme  20-24 ans            239255
1        2022  Auvergne-Rhône-Alpes  Homme  25-29 ans            229782
2        2022  Auvergne-Rhône-Alpes  Homme  30-34 ans            247295
3     

## 6. Export du fichier préparé

In [6]:
insee_long.to_csv(output_path, index=False, encoding="utf-8")
print(f"Fichier créé : {output_path}")

Fichier créé : data/processed/insee_population_region_sexe_age_2022_2025_long.csv


## Résultat

La transformation produit **1 008 lignes** :

- 4 années ;
- 14 régions ;
- 2 genres ;
- 9 tranches d’âge.

Le fichier obtenu est ensuite chargé dans Snowflake et transformé avec dbt.